# Lab – Step 1: Build the Initial Private Knowledge Base

**Domain: Music** – Artists, Albums, Genres, Awards, Record Labels, and Countries

We will:
1. Install / import dependencies
2. Define namespace prefixes
3. Populate the graph programmatically (simulating data collected from an API)
4. Validate volume (≥ 100 triples, ≥ 50 entities)
5. Serialize to Turtle (`.ttl`) for downstream use

In [2]:
from rdflib import Graph, Namespace, URIRef, Literal, RDF, RDFS, OWL, XSD
from rdflib.namespace import FOAF, DC

# ── Custom namespace (our private KB base URI) ──────────────────────────────
MUS  = Namespace("http://example.org/music/")      # entities
PROP = Namespace("http://example.org/music/prop/") # properties
SCH  = Namespace("https://schema.org/")            # reuse Schema.org where sensible

g = Graph()
g.bind("mus",  MUS)
g.bind("prop", PROP)
g.bind("sch",  SCH)
g.bind("foaf", FOAF)
g.bind("dc",   DC)
g.bind("xsd",  XSD)
g.bind("owl",  OWL)
g.bind("rdfs", RDFS)

print("Graph initialised.")

Graph initialised.


## 2. Ontology / Class declarations

Good practice: declare your classes and properties explicitly.

In [3]:
# ── Classes ──────────────────────────────────────────────────────────────────
classes = [
    (MUS.Artist,      "A musical artist (solo or group)"),
    (MUS.Album,       "A studio or live album"),
    (MUS.Genre,       "A music genre"),
    (MUS.Award,       "A music industry award"),
    (MUS.RecordLabel, "A record label company"),
    (MUS.Country,     "A country of origin"),
]
for cls, label in classes:
    g.add((cls, RDF.type,   OWL.Class))
    g.add((cls, RDFS.label, Literal(label)))

# ── Object Properties ─────────────────────────────────────────────────────
obj_props = [
    (PROP.hasGenre,        "Artist or album has genre"),
    (PROP.wonAward,        "Artist won award"),
    (PROP.releasedAlbum,   "Artist released album"),
    (PROP.signedTo,        "Artist signed to record label"),
    (PROP.originatesFrom,  "Artist originates from country"),
    (PROP.collaboratedWith,"Artist collaborated with another artist"),
    (PROP.influencedBy,    "Artist was influenced by another artist"),
    (PROP.producedBy,      "Album produced by artist"),
    (PROP.partOfGenre,     "Genre is a subgenre of another genre"),
]
for prop, label in obj_props:
    g.add((prop, RDF.type,   OWL.ObjectProperty))
    g.add((prop, RDFS.label, Literal(label)))

# ── Datatype Properties ───────────────────────────────────────────────────
data_props = [
    (PROP.birthYear,    XSD.gYear,    "Year of birth"),
    (PROP.activeFrom,   XSD.gYear,    "Year career started"),
    (PROP.releaseYear,  XSD.gYear,    "Year album released"),
    (PROP.albumRating,  XSD.decimal,  "Critical rating 0-10"),
    (PROP.foundedYear,  XSD.gYear,    "Year label founded"),
]
for prop, dtype, label in data_props:
    g.add((prop, RDF.type,      OWL.DatatypeProperty))
    g.add((prop, RDFS.label,    Literal(label)))
    g.add((prop, RDFS.range,    dtype))

print(f"Schema triples added. Graph size: {len(g)}")

Schema triples added. Graph size: 45


## 3. Populate Entities

### 3a. Genres

In [4]:
genres = {
    "Rock":          "Rock music",
    "Pop":           "Pop music",
    "Jazz":          "Jazz music",
    "HipHop":        "Hip-hop music",
    "Electronic":    "Electronic music",
    "Classical":     "Classical music",
    "RnB":           "Rhythm and Blues",
    "AlternativeRock":"Alternative Rock",
    "Soul":          "Soul music",
    "Folk":          "Folk music",
}

subgenres = [
    ("AlternativeRock", "Rock"),
    ("Soul", "RnB"),
]

for gname, glabel in genres.items():
    uri = MUS[gname]
    g.add((uri, RDF.type,        MUS.Genre))
    g.add((uri, RDFS.label,      Literal(glabel)))

for sub, parent in subgenres:
    g.add((MUS[sub], PROP.partOfGenre, MUS[parent]))

print(f"After genres: {len(g)} triples")

After genres: 67 triples


### 3b. Countries

In [5]:
countries = ["USA", "UK", "Canada", "Australia", "Jamaica", "France", "Nigeria"]

for c in countries:
    uri = MUS[c]
    g.add((uri, RDF.type,   MUS.Country))
    g.add((uri, RDFS.label, Literal(c)))

print(f"After countries: {len(g)} triples")

After countries: 81 triples


### 3c. Record Labels

In [6]:
labels_data = [
    ("ColumbiaRecords",  "Columbia Records",  1887, "USA"),
    ("UniversalMusic",   "Universal Music",   1934, "USA"),
    ("SonyMusicUK",      "Sony Music UK",     1929, "UK"),
    ("DefJamRecordings", "Def Jam Recordings",1983, "USA"),
    ("BlueNote",         "Blue Note Records", 1939, "USA"),
    ("XLRecordings",     "XL Recordings",     1989, "UK"),
]

for lid, lname, founded, country in labels_data:
    uri = MUS[lid]
    g.add((uri, RDF.type,        MUS.RecordLabel))
    g.add((uri, RDFS.label,      Literal(lname)))
    g.add((uri, PROP.foundedYear,Literal(founded, datatype=XSD.gYear)))
    g.add((uri, PROP.originatesFrom, MUS[country]))

print(f"After labels: {len(g)} triples")

After labels: 105 triples


### 3d. Awards

In [7]:
awards_data = [
    ("GrammyBestAlbum",     "Grammy Award for Album of the Year"),
    ("GrammyBestArtist",    "Grammy Award for Best New Artist"),
    ("GrammyBestRnB",       "Grammy Award for Best R&B Album"),
    ("GrammyBestRap",       "Grammy Award for Best Rap Album"),
    ("BritAwardArtist",     "BRIT Award for British Artist of the Year"),
    ("MercuryPrize",        "Mercury Prize"),
    ("MTVVideoAward",       "MTV Video Music Award"),
    ("BillboardMusicAward", "Billboard Music Award"),
]

for aid, alabel in awards_data:
    uri = MUS[aid]
    g.add((uri, RDF.type,   MUS.Award))
    g.add((uri, RDFS.label, Literal(alabel)))

print(f"After awards: {len(g)} triples")

After awards: 121 triples


### 3e. Artists

In [8]:
# Format: (id, label, birthYear, activeFrom, country, label_id, [genres], [awards])
artists_data = [
    ("BeyonceKnowles",  "Beyoncé Knowles",   1981, 1997, "USA",       "ColumbiaRecords",  ["RnB","Pop"],            ["GrammyBestAlbum","GrammyBestRnB","BillboardMusicAward"]),
    ("MichaelJackson",  "Michael Jackson",   1958, 1964, "USA",       "SonyMusicUK",      ["Pop","RnB","Soul"],     ["GrammyBestAlbum","GrammyBestArtist","MTVVideoAward"]),
    ("DavidBowie",      "David Bowie",        1947, 1964, "UK",        "ColumbiaRecords",  ["Rock","AlternativeRock"],["BritAwardArtist"]),
    ("ArethaFranklin",  "Aretha Franklin",    1942, 1956, "USA",       "ColumbiaRecords",  ["Soul","RnB","Jazz"],    ["GrammyBestAlbum","GrammyBestRnB"]),
    ("NingeniaClapton", "Eric Clapton",       1945, 1963, "UK",        "UniversalMusic",   ["Rock"],                 ["GrammyBestAlbum"]),
    ("JayZ",            "Jay-Z",              1969, 1989, "USA",       "DefJamRecordings", ["HipHop"],               ["GrammyBestRap","BillboardMusicAward"]),
    ("KendrickLamar",   "Kendrick Lamar",     1987, 2003, "USA",       "UniversalMusic",   ["HipHop"],               ["GrammyBestAlbum","GrammyBestRap"]),
    ("Adele",           "Adele Laurie Blue",  1988, 2006, "UK",        "XLRecordings",     ["Pop","Soul"],           ["GrammyBestAlbum","GrammyBestArtist","BritAwardArtist"]),
    ("DrakeBell",       "Drake",              1986, 2001, "Canada",    "UniversalMusic",   ["HipHop","RnB"],         ["GrammyBestRap","BillboardMusicAward"]),
    ("TaylorSwift",     "Taylor Swift",       1989, 2004, "USA",       "UniversalMusic",   ["Pop","Folk"],           ["GrammyBestAlbum","GrammyBestArtist","MTVVideoAward"]),
    ("RadioheadBand",   "Radiohead",          1985, 1985, "UK",        "XLRecordings",     ["AlternativeRock","Electronic"],["MercuryPrize"]),
    ("NinaSimone",      "Nina Simone",        1933, 1954, "USA",       "ColumbiaRecords",  ["Jazz","Soul","Folk"],   ["GrammyBestAlbum"]),
    ("BobMarley",       "Bob Marley",         1945, 1963, "Jamaica",   "UniversalMusic",   ["Folk"],                 ["BillboardMusicAward"]),
    ("DaftPunk",        "Daft Punk",          1987, 1993, "France",    "UniversalMusic",   ["Electronic","Pop"],     ["GrammyBestAlbum","MTVVideoAward"]),
    ("BurnaFire",       "Burna Boy",          1991, 2011, "Nigeria",   "UniversalMusic",   ["RnB","Pop"],            ["GrammyBestRnB"]),
    ("PrinceRogers",    "Prince",             1958, 1975, "USA",       "UniversalMusic",   ["Pop","RnB","Rock"],     ["GrammyBestAlbum","MTVVideoAward"]),
    ("BilliEilish",     "Billie Eilish",      2001, 2015, "USA",       "UniversalMusic",   ["Pop","Electronic"],     ["GrammyBestAlbum","GrammyBestArtist"]),
    ("FrankOcean",      "Frank Ocean",        1987, 2009, "USA",       "DefJamRecordings", ["RnB","HipHop"],         ["GrammyBestAlbum"]),
    ("PaulMcCartney",   "Paul McCartney",     1942, 1960, "UK",        "ColumbiaRecords",  ["Rock","Pop"],           ["GrammyBestAlbum","BritAwardArtist"]),
    ("MilesDavis",      "Miles Davis",        1926, 1944, "USA",       "ColumbiaRecords",  ["Jazz"],                 ["GrammyBestAlbum"]),
]

for aid, alabel, born, active, country, label_id, genre_list, award_list in artists_data:
    uri = MUS[aid]
    g.add((uri, RDF.type,           MUS.Artist))
    g.add((uri, RDFS.label,         Literal(alabel)))
    g.add((uri, FOAF.name,          Literal(alabel)))
    g.add((uri, PROP.birthYear,     Literal(born,   datatype=XSD.gYear)))
    g.add((uri, PROP.activeFrom,    Literal(active, datatype=XSD.gYear)))
    g.add((uri, PROP.originatesFrom,MUS[country]))
    g.add((uri, PROP.signedTo,      MUS[label_id]))
    for genre in genre_list:
        g.add((uri, PROP.hasGenre,  MUS[genre]))
    for award in award_list:
        g.add((uri, PROP.wonAward,  MUS[award]))

print(f"After artists: {len(g)} triples")

After artists: 336 triples


### 3f. Collaborations & Influences

In [9]:
collaborations = [
    ("BeyonceKnowles", "JayZ"),
    ("KendrickLamar",  "DrakeBell"),
    ("DaftPunk",       "BeyonceKnowles"),
    ("BurnaFire",      "BeyonceKnowles"),
    ("PaulMcCartney",  "MichaelJackson"),
    ("JayZ",           "FrankOcean"),
]

influences = [
    ("BeyonceKnowles", "MichaelJackson"),
    ("BeyonceKnowles", "ArethaFranklin"),
    ("MichaelJackson", "JamesB"),  # placeholder to show external ref
    ("KendrickLamar",  "NinaSimone"),
    ("Adele",          "ArethaFranklin"),
    ("BilliEilish",    "FrankOcean"),
    ("FrankOcean",     "MilesDavis"),
    ("RadioheadBand",  "DavidBowie"),
    ("DrakeBell",      "JayZ"),
    ("TaylorSwift",    "PaulMcCartney"),
]

# Add JamesB as a stub entity (shows KB can have seed entities to expand later)
g.add((MUS["JamesB"], RDF.type,   MUS.Artist))
g.add((MUS["JamesB"], RDFS.label, Literal("James Brown")))
g.add((MUS["JamesB"], PROP.hasGenre, MUS["Soul"]))
g.add((MUS["JamesB"], PROP.originatesFrom, MUS["USA"]))

for a1, a2 in collaborations:
    g.add((MUS[a1], PROP.collaboratedWith, MUS[a2]))

for a1, a2 in influences:
    g.add((MUS[a1], PROP.influencedBy, MUS[a2]))

print(f"After relations: {len(g)} triples")

After relations: 356 triples


### 3g. Albums

In [10]:
# Format: (id, label, year, artist_id, [genres], rating, award_or_None)
albums_data = [
    ("Lemonade",       "Lemonade",               2016, "BeyonceKnowles", ["RnB","Pop"],            9.0, "GrammyBestAlbum"),
    ("Thriller",       "Thriller",               1982, "MichaelJackson", ["Pop","RnB"],             9.8, "GrammyBestAlbum"),
    ("Ziggy",          "Ziggy Stardust",          1972, "DavidBowie",     ["Rock","AlternativeRock"],9.2, None),
    ("I_Never_Loved",  "I Never Loved a Man",     1967, "ArethaFranklin", ["Soul","RnB"],            9.5, "GrammyBestRnB"),
    ("TPAB",           "To Pimp a Butterfly",     2015, "KendrickLamar", ["HipHop"],                9.7, "GrammyBestRap"),
    ("21",             "21",                      2011, "Adele",          ["Pop","Soul"],             9.0, "GrammyBestAlbum"),
    ("KindOfBlue",     "Kind of Blue",            1959, "MilesDavis",     ["Jazz"],                  9.9, "GrammyBestAlbum"),
    ("BlondeAlbum",    "Blonde",                  2016, "FrankOcean",     ["RnB","HipHop"],          9.6, None),
    ("Discovery",      "Discovery",               2001, "DaftPunk",       ["Electronic","Pop"],       9.1, "GrammyBestAlbum"),
    ("OKComputer",     "OK Computer",             1997, "RadioheadBand", ["AlternativeRock"],        9.8, "MercuryPrize"),
    ("Legend",         "Legend",                  1984, "BobMarley",      ["Folk"],                  8.9, None),
    ("FearOfGodAlbum", "WHEN WE ALL FALL ASLEEP", 2019, "BilliEilish",    ["Pop","Electronic"],       8.7, "GrammyBestAlbum"),
    ("Blueprint",      "The Blueprint",           2001, "JayZ",           ["HipHop"],                9.3, "GrammyBestRap"),
    ("PurpleRain",     "Purple Rain",             1984, "PrinceRogers",   ["Pop","RnB"],             9.4, "GrammyBestAlbum"),
    ("AbbeyRoad",      "Abbey Road",              1969, "PaulMcCartney",  ["Rock","Pop"],             9.7, "GrammyBestAlbum"),
]

for alb_id, alb_label, year, artist_id, genres, rating, award in albums_data:
    uri = MUS[alb_id]
    g.add((uri, RDF.type,          MUS.Album))
    g.add((uri, RDFS.label,        Literal(alb_label)))
    g.add((uri, DC.title,          Literal(alb_label)))
    g.add((uri, PROP.releaseYear,  Literal(year,   datatype=XSD.gYear)))
    g.add((uri, PROP.albumRating,  Literal(rating, datatype=XSD.decimal)))
    g.add((uri, PROP.producedBy,   MUS[artist_id]))
    g.add((MUS[artist_id], PROP.releasedAlbum, uri))
    for genre in genres:
        g.add((uri, PROP.hasGenre, MUS[genre]))
    if award:
        g.add((uri, PROP.wonAward, MUS[award]))

print(f"After albums: {len(g)} triples")

After albums: 498 triples


## 4. Validation

In [11]:
# ── Count triples ─────────────────────────────────────────────────────────
total_triples = len(g)

# ── Count distinct entities (subjects that are not blank nodes) ───────────
entities = set()
for s, p, o in g:
    if isinstance(s, URIRef):
        entities.add(s)
    if isinstance(o, URIRef):
        entities.add(o)

# Filter to only our MUS namespace (exclude schema.org, owl, etc.)
mus_entities = {e for e in entities if str(e).startswith(str(MUS))}

print("=" * 50)
print(f"Total triples   : {total_triples}")
print(f"Distinct MUS URIs: {len(mus_entities)}")
print("=" * 50)

if total_triples >= 100:
    print("✅  PASS: ≥ 100 triples")
else:
    print("❌  FAIL: fewer than 100 triples")

if len(mus_entities) >= 50:
    print("✅  PASS: ≥ 50 entities")
else:
    print("❌  FAIL: fewer than 50 entities")

Total triples   : 498
Distinct MUS URIs: 87
✅  PASS: ≥ 100 triples
✅  PASS: ≥ 50 entities


## 5. Entity breakdown by class

In [12]:
from collections import Counter

type_counts = Counter()
for s, p, o in g.triples((None, RDF.type, None)):
    if isinstance(s, URIRef) and str(s).startswith(str(MUS)):
        type_counts[str(o).split("/")[-1]] += 1

print("Entity counts by class:")
for cls, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {cls:<20} {count}")

Entity counts by class:
  Artist               21
  Album                15
  Genre                10
  owl#ObjectProperty   9
  Award                8
  Country              7
  owl#Class            6
  RecordLabel          6
  owl#DatatypeProperty 5


## 6. Quick SPARQL smoke-test on the local graph

In [13]:
# Which artists won the Grammy Album of the Year?
q = """
PREFIX mus:  <http://example.org/music/>
PREFIX prop: <http://example.org/music/prop/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?artistLabel WHERE {
    ?artist a mus:Artist ;
            prop:wonAward mus:GrammyBestAlbum ;
            rdfs:label ?artistLabel .
}
ORDER BY ?artistLabel
"""
print("Artists who won Grammy Best Album:")
for row in g.query(q):
    print(" -", row.artistLabel)

Artists who won Grammy Best Album:
 - Adele Laurie Blue
 - Aretha Franklin
 - Beyoncé Knowles
 - Billie Eilish
 - Daft Punk
 - Eric Clapton
 - Frank Ocean
 - Kendrick Lamar
 - Michael Jackson
 - Miles Davis
 - Nina Simone
 - Paul McCartney
 - Prince
 - Taylor Swift


In [14]:
# Which albums received a rating > 9.5 ?
q2 = """
PREFIX mus:  <http://example.org/music/>
PREFIX prop: <http://example.org/music/prop/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>

SELECT ?albumLabel ?rating WHERE {
    ?album a mus:Album ;
           rdfs:label ?albumLabel ;
           prop:albumRating ?rating .
    FILTER(?rating > 9.5)
}
ORDER BY DESC(?rating)
"""
print("Albums rated > 9.5:")
for row in g.query(q2):
    print(f" - {row.albumLabel}  ({row.rating})")

Albums rated > 9.5:
 - Kind of Blue  (9.9)
 - Thriller  (9.8)
 - OK Computer  (9.8)
 - To Pimp a Butterfly  (9.7)
 - Abbey Road  (9.7)
 - Blonde  (9.6)


## 7. Serialize to Turtle

In [15]:
output_path = "music_kb.ttl"
g.serialize(destination=output_path, format="turtle")
print(f"Knowledge base saved → {output_path}")

# Preview first 40 lines
with open(output_path) as f:
    lines = f.readlines()
print(f"Total lines in file: {len(lines)}")
print("\n--- First 40 lines ---")
print("".join(lines[:40]))

Knowledge base saved → music_kb.ttl
Total lines in file: 593

--- First 40 lines ---
@prefix dc: <http://purl.org/dc/elements/1.1/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix mus: <http://example.org/music/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix prop: <http://example.org/music/prop/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

mus:Australia a mus:Country ;
    rdfs:label "Australia" .

mus:BlueNote a mus:RecordLabel ;
    rdfs:label "Blue Note Records" ;
    prop:foundedYear "1939"^^xsd:gYear ;
    prop:originatesFrom mus:USA .

mus:BurnaFire a mus:Artist ;
    rdfs:label "Burna Boy" ;
    prop:activeFrom "2011"^^xsd:gYear ;
    prop:birthYear "1991"^^xsd:gYear ;
    prop:collaboratedWith mus:BeyonceKnowles ;
    prop:hasGenre mus:Pop,
        mus:RnB ;
    prop:originatesFrom mus:Nigeria ;
    prop:signedTo mus:UniversalMusic ;
    prop:wonAward mus:GrammyBestRnB ;
    foaf:name "Burna Bo

In [1]:
!jupyter nbconvert --to html step1_build_knowledge_base.ipynb


[NbConvertApp] Converting notebook step1_build_knowledge_base.ipynb to html
[NbConvertApp] Writing 367220 bytes to step1_build_knowledge_base.html
